# QCT POSCAR Generator with ZPE Constraints

This notebook builds molecule + surface initial configurations for QCT trajectories. It uses vibrational frequencies and eigenvectors extracted from an isolated-molecule VASP calculation for ZPE initialization, and snapshots from a thermalized VASP surface trajectory. No dynamics are run here: the notebook only writes POSCAR files for random initial-condition samples.


## Workflow

1. Load the optimized isolated-molecule geometry directly from `vasprun.xml`.
2. Read frequencies in cm^-1 and Cartesian eigenvectors from the same molecule `vasprun.xml`.
3. Load a surface snapshot ensemble directly from `vasprun.xml`.
4. For each requested configuration:
   - Initialize the molecule with semiclassical ZPE corrections using a random phase.
   - Apply a random rotation and place the molecule above the surface with a random XY translation.
   - Merge the two atom sets and write `POSCAR-*`.


In [ ]:
import json
import re
import xml.etree.ElementTree as ET
from pathlib import Path
from typing import Optional, Sequence, Tuple

import numpy as np
from ase import units
from ase.atoms import Atoms
from ase.io import read, write

In [ ]:
# --- User parameters ---
PATHS = {
    "molecule_vasprun": Path("inputs/molecule/SO2/vasprun_SO2.xml"),
    "vasp_modes_npz": Path("inputs/molecule/SO2/vibrational_modes-SO2.npz"),
    "surface_vasprun": Path("inputs/surface/HOPG_therm/vasprun-500K.xml"),
    "output_dir": Path("outputs/qct_poscars"),
}

SURFACE_SAMPLING = {
    "format": "vasp-xml",     # vasprun.xml read with ASE
    "index": "::10",          # ASE slice syntax, e.g. '::5' for one frame every five frames
    "keep_velocities": True,    # True if the file already contains useful velocities
    "POTIM_fs": 1.0,            # time step between frames for velocity estimation if needed
}

GENERATION = {
    "n_configurations": 10,
    "master_seed": 20240331,
    "surface_distance_A": 7.0,  # fixed distance between molecule COM and the top surface plane
    "min_clearance": 1.3,       # anti-overlap safety; may slightly increase the effective distance
    "incident_energy_eV": 2,  # translational incident kinetic energy added to the molecule
    "incident_direction": (0.0, 0.0, -1.0),  # incidence direction toward the surface
    "xy_mode": "random",       # 'random' or 'center'
}

ZPE_SETTINGS = {
    "enabled": True,         # Enable ZPE initialization from VASP normalmode_eigenvals (THz^2 -> cm^-1)
    "freq_key": "freqs_cm1",
    "eigvec_key": "eigvecs_cart",
    "v_quantum": None,        # None -> [0] * n_vib
    "seed_offset": 0,         # offset to decorrelate ZPE draws
}

ROTATION_SETTINGS = {
    "temperature_K": 0.0,    # 0.0 or "0K" -> no added rotational energy
    "seed_offset": 100000,   # offset to decorrelate rotational and ZPE draws
}

PATHS["output_dir"].mkdir(parents=True, exist_ok=True)
print(json.dumps({k: str(v) for k, v in PATHS.items()}, indent=2))
print(f"Vibrational modes read from {PATHS['molecule_vasprun']}")
print(f".npz cache written to {PATHS['vasp_modes_npz']}")

### VASP Vibrational Data Preparation from `vasprun.xml`

The notebook reads vibrational frequencies and Cartesian eigenvectors directly from the molecule `vasprun.xml` with `pymatgen`.

It then automatically writes a cache file containing:
- `freqs_cm1`: a 1D array of length 3N with frequencies in cm^-1.
- `eigvecs_cart`: an array with shape (3N, N, 3) or (3N, 3N) containing normalized Cartesian eigenvectors.

You only need to provide the VASP output files. Make sure the molecule `vasprun.xml` contains vibrational modes (`IBRION=5` or `IBRION=6`).


In [ ]:
# --- ZPE helpers ---
EV_TO_J = 1.602176634e-19
AMU_TO_KG = 1.66053906660e-27
ANG_TO_M = 1.0e-10
FS_TO_S = 1.0e-15
C_CM_S = 2.99792458e10
THZ_TO_CM1 = 33.35640951981521
HBAR_EV_FS = 0.6582119514
LAMBDA_TO_OMEGA2_FS2 = (EV_TO_J / (AMU_TO_KG * ANG_TO_M**2)) * (FS_TO_S**2)
KCONV = AMU_TO_KG * (ANG_TO_M / FS_TO_S)**2 / EV_TO_J
KB_EV_K = 8.617333262145e-5


def freq_cm1_to_omega_fs(freq_cm1: float) -> float:
    omega_si = 2.0 * np.pi * C_CM_S * freq_cm1
    return omega_si * FS_TO_S


def is_linear_molecule(atoms: Atoms, tol: float = 1e-8) -> bool:
    pos = atoms.get_positions()
    pos_centered = pos - pos.mean(axis=0)
    _, s, _ = np.linalg.svd(pos_centered, full_matrices=False)
    if len(s) < 2:
        return True
    return s[1] < tol


def get_vibrational_mode_info(atoms: Atoms) -> dict:
    nat = len(atoms)
    ndof = 3 * nat
    linear = is_linear_molecule(atoms)
    n_zero = 5 if linear else 6
    return {"nat": nat, "ndof": ndof, "linear": linear, "n_zero": n_zero, "n_vib": ndof - n_zero}


def normalize_isolated_molecule(atoms: Atoms) -> Atoms:
    mol = atoms.copy()
    ref = mol.positions[0].copy()
    new_pos = np.zeros((len(mol), 3))
    new_pos[0] = ref
    for i in range(1, len(mol)):
        new_pos[i] = ref + mol.get_distance(0, i, mic=True, vector=True)
    mol.set_positions(new_pos)
    mol.positions -= mol.get_center_of_mass()
    mol.set_pbc(False)
    mol.set_cell(np.zeros((3, 3)))
    return mol


def load_structure_from_vasprun(path: Path, index: int = -1) -> Atoms:
    atoms = read(path, index=index, format="vasp-xml")
    if not isinstance(atoms, Atoms):
        raise ValueError("The molecule vasprun.xml must provide a single ASE geometry.")
    return normalize_isolated_molecule(atoms)


def load_modes_from_vasprun(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    try:
        from pymatgen.io.vasp.outputs import Vasprun
    except ImportError as exc:
        raise ImportError("Install pymatgen (pip install pymatgen) to read vasprun.xml directly.") from exc
    vrun = Vasprun(str(path), parse_dos=False, parse_eigen=False)
    eigvals = getattr(vrun, "normalmode_eigenvals", None)
    eig = getattr(vrun, "normalmode_eigenvecs", None)
    if eigvals is None or eig is None:
        raise ValueError("The vasprun.xml file does not contain vibrational modes (IBRION=5/6 calculation required).")
    eigvals = np.asarray(eigvals, dtype=float)
    eig = np.asarray(eig, dtype=float)
    order = np.argsort(np.abs(eigvals))
    freqs_cm1 = np.sqrt(np.abs(eigvals[order])) * THZ_TO_CM1
    eig = eig[order]
    return freqs_cm1, eig


def save_modes_npz(path: Path, freqs_cm1: np.ndarray, eigvecs_cart: np.ndarray) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez(path, freqs_cm1=np.asarray(freqs_cm1, dtype=float), eigvecs_cart=np.asarray(eigvecs_cart, dtype=float))


def reshape_eigvecs(eig: np.ndarray, nat: int) -> np.ndarray:
    if eig.ndim == 3:
        n_modes, n_atoms, n_dir = eig.shape
        if n_atoms != nat or n_dir != 3:
            raise ValueError("Eigenvector dimensions do not match the molecule.")
        eig = eig.reshape(n_modes, n_atoms * n_dir)
    if eig.ndim != 2:
        raise ValueError("Unsupported eigenvector format.")
    n_modes, ndof = eig.shape
    if ndof != 3 * nat or n_modes != 3 * nat:
        raise ValueError("Expected 3N modes and 3N components per mode.")
    return eig.T


def build_mass_weighted_modes(atoms: Atoms, eig_cart: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    nat = len(atoms)
    eig_matrix = reshape_eigvecs(eig_cart, nat)
    masses = np.repeat(atoms.get_masses(), 3)
    sqrt_m = np.sqrt(masses)
    eig_mass = eig_matrix / sqrt_m[:, None]
    return eig_mass, masses

def semiclassical_zpe_external(atoms: Atoms, freqs_cm1: np.ndarray, eig_mass: np.ndarray,
                                masses_dof: np.ndarray, v_quantum: Optional[Sequence[int]] = None,
                                seed: Optional[int] = None):
    info = get_vibrational_mode_info(atoms)
    ndof = info["ndof"]
    n_zero = info["n_zero"]
    n_vib = info["n_vib"]
    if freqs_cm1.size != ndof:
        raise ValueError("Number of frequencies is incompatible with 3N.")
    if eig_mass.shape != (ndof, ndof):
        raise ValueError("The mode matrix must be square (3N, 3N).")
    if v_quantum is None:
        v_quantum = [0] * n_vib
    if len(v_quantum) != n_vib:
        raise ValueError("v_quantum must contain {n_vib} elements.".format(n_vib=n_vib))
    rng = np.random.default_rng(seed)
    delta_Q = np.zeros(ndof)
    delta_Qdot = np.zeros(ndof)
    for k, vk in enumerate(v_quantum):
        mode_idx = n_zero + k
        freq = freqs_cm1[mode_idx]
        if freq <= 0:
            continue
        omega_fs = freq_cm1_to_omega_fs(freq)
        lam = (omega_fs ** 2) / LAMBDA_TO_OMEGA2_FS2
        E_k = HBAR_EV_FS * omega_fs * (vk + 0.5)
        gamma = rng.uniform(0.0, 2.0 * np.pi)
        Q_amp = np.sqrt(2.0 * E_k / lam)
        Qdot_amp = np.sqrt(2.0 * E_k / KCONV)
        delta_Q[mode_idx] = Q_amp * np.cos(gamma)
        delta_Qdot[mode_idx] = -Qdot_amp * np.sin(gamma)
    delta_pos_flat = eig_mass @ delta_Q
    delta_vel_flat = eig_mass @ delta_Qdot
    delta_pos = delta_pos_flat.reshape(len(atoms), 3)
    delta_vel = delta_vel_flat.reshape(len(atoms), 3)
    return delta_pos, delta_vel


def remove_com_and_angular_momentum(atoms: Atoms, delta_pos: np.ndarray, delta_vel: np.ndarray):
    masses = atoms.get_masses()
    total_mass = masses.sum()
    v_cm = (masses[:, None] * delta_vel).sum(axis=0) / total_mass
    delta_vel -= v_cm[None, :]
    pos_eq = atoms.get_positions()
    L = np.zeros(3)
    for i in range(len(atoms)):
        L += masses[i] * np.cross(pos_eq[i] + delta_pos[i], delta_vel[i])
    r_cm = (masses[:, None] * pos_eq).sum(axis=0) / total_mass
    I = np.zeros((3, 3))
    for i in range(len(atoms)):
        r = pos_eq[i] - r_cm
        I += masses[i] * (np.dot(r, r) * np.eye(3) - np.outer(r, r))
    try:
        omega = np.linalg.solve(I, L)
    except np.linalg.LinAlgError:
        omega = np.zeros(3)
    for i in range(len(atoms)):
        r_i = pos_eq[i] + delta_pos[i] - r_cm
        delta_vel[i] -= np.cross(omega, r_i)
    return delta_pos, delta_vel


def rotational_temperature_to_kelvin(value) -> float:
    if isinstance(value, str):
        text = value.strip().lower()
        if text.endswith("k"):
            text = text[:-1]
        if not text:
            raise ValueError("temperature_K cannot be empty.")
        value = float(text)
    temp_K = float(value)
    if temp_K < 0.0:
        raise ValueError("Rotational temperature must be non-negative.")
    return temp_K


def inertia_tensor_about_com(atoms: Atoms) -> tuple[np.ndarray, np.ndarray]:
    masses = atoms.get_masses()
    pos = atoms.get_positions()
    r_cm = (masses[:, None] * pos).sum(axis=0) / masses.sum()
    rel = pos - r_cm
    inertia = np.zeros((3, 3))
    for mass, r in zip(masses, rel):
        inertia += mass * (np.dot(r, r) * np.eye(3) - np.outer(r, r))
    return inertia, rel


def add_rotational_energy_from_temperature(mol: Atoms, temperature_K: float, seed: Optional[int] = None,
                                           tol: float = 1e-12) -> float:
    temp_K = rotational_temperature_to_kelvin(temperature_K)
    if temp_K == 0.0:
        return 0.0

    inertia, rel = inertia_tensor_about_com(mol)
    eigvals, eigvecs = np.linalg.eigh(inertia)
    active = eigvals > tol
    if not np.any(active):
        return 0.0

    rng = np.random.default_rng(seed)
    omega_principal = np.zeros(3)
    sigma = np.sqrt(KB_EV_K * temp_K / eigvals[active])
    omega_principal[active] = rng.normal(loc=0.0, scale=sigma)
    omega = eigvecs @ omega_principal

    delta_vel = np.cross(omega[None, :], rel)
    vel = mol.get_velocities()
    if vel is None:
        vel = np.zeros((len(mol), 3))
    mol.set_velocities(vel + delta_vel / units.fs)

    energy_eV = 0.5 * np.sum(eigvals[active] * omega_principal[active] ** 2)
    return float(energy_eV)


def theoretical_vibrational_energy(freqs_cm1: np.ndarray, v_quantum: Sequence[int], n_zero: int) -> float:
    energy = 0.0
    for k, vk in enumerate(v_quantum):
        freq = freqs_cm1[n_zero + k]
        if freq <= 0:
            continue
        omega_fs = freq_cm1_to_omega_fs(freq)
        energy += HBAR_EV_FS * omega_fs * (vk + 0.5)
    return float(energy)


def project_normal_coordinates(delta: np.ndarray, eig_mass: np.ndarray, masses_dof: np.ndarray) -> np.ndarray:
    delta_flat = np.asarray(delta, dtype=float).reshape(-1)
    return eig_mass.T @ (masses_dof * delta_flat)


def verify_zpe(atoms: Atoms, delta_pos: np.ndarray, delta_vel: np.ndarray, freqs_cm1: np.ndarray,
               eig_mass: np.ndarray, masses_dof: np.ndarray, v_quantum: Sequence[int],
               verbose: bool = False) -> dict:
    masses = atoms.get_masses()
    kinetic_cart = 0.5 * np.sum(masses[:, None] * delta_vel**2) * KCONV
    info = get_vibrational_mode_info(atoms)
    n_zero = info["n_zero"]
    target_energy = theoretical_vibrational_energy(freqs_cm1, v_quantum, n_zero)
    Q = project_normal_coordinates(delta_pos, eig_mass, masses_dof)
    Qdot = project_normal_coordinates(delta_vel, eig_mass, masses_dof)
    mode_kinetic = 0.5 * (Qdot**2) * KCONV
    mode_potential = np.zeros_like(mode_kinetic)
    for mode_idx in range(n_zero, len(freqs_cm1)):
        freq = freqs_cm1[mode_idx]
        if freq <= 0:
            continue
        omega_fs = freq_cm1_to_omega_fs(freq)
        lam = (omega_fs ** 2) / LAMBDA_TO_OMEGA2_FS2
        mode_potential[mode_idx] = 0.5 * lam * (Q[mode_idx] ** 2)
    mode_total = mode_kinetic + mode_potential
    mode_indices = list(range(n_zero, len(freqs_cm1)))
    mode_details = []
    for local_idx, mode_idx in enumerate(mode_indices):
        mode_details.append({
            "mode_index": int(mode_idx),
            "vibrational_mode_index": int(local_idx),
            "frequency_cm1": float(freqs_cm1[mode_idx]),
            "quantum": int(v_quantum[local_idx]),
            "kinetic_eV": float(mode_kinetic[mode_idx]),
            "potential_eV": float(mode_potential[mode_idx]),
            "total_eV": float(mode_total[mode_idx]),
        })
    report = {
        "target_total_eV": float(target_energy),
        "cartesian_kinetic_eV": float(kinetic_cart),
        "modal_kinetic_eV": float(mode_kinetic[n_zero:].sum()),
        "modal_potential_eV": float(mode_potential[n_zero:].sum()),
        "modal_total_eV": float(mode_total[n_zero:].sum()),
        "external_leak_eV": float(mode_total[:n_zero].sum()),
        "mode_energies_eV": mode_total[n_zero:].tolist(),
        "mode_details": mode_details,
    }
    if verbose:
        print(f"Target E_ZPE        : {report['target_total_eV']:.6f} eV")
        print(f"E_vib cinetique    : {report['modal_kinetic_eV']:.6f} eV")
        print(f"E_vib potentielle  : {report['modal_potential_eV']:.6f} eV")
        print(f"E_vib totale       : {report['modal_total_eV']:.6f} eV")
        print(f"Fuite ext. (TR)    : {report['external_leak_eV']:.6e} eV")
    return report


class MoleculeZPEInitializer:
    def __init__(self, molecule_eq: Atoms, freqs_cm1: np.ndarray, eigvecs_cart: np.ndarray,
                 v_quantum: Optional[Sequence[int]] = None, seed_offset: int = 0):
        self.template = molecule_eq.copy()
        self.freqs_cm1 = np.array(freqs_cm1, dtype=float)
        eig_mass, masses = build_mass_weighted_modes(self.template, eigvecs_cart)
        self.eig_mass = eig_mass
        self.masses_dof = masses
        info = get_vibrational_mode_info(self.template)
        self.n_vib = info["n_vib"]
        self.v_quantum = v_quantum if v_quantum is not None else [0] * self.n_vib
        self.seed_offset = seed_offset

    def sample(self, seed: int) -> Atoms:
        mol = self.template.copy()
        delta_pos, delta_vel = semiclassical_zpe_external(
            mol,
            self.freqs_cm1,
            self.eig_mass,
            self.masses_dof,
            v_quantum=self.v_quantum,
            seed=seed + self.seed_offset,
        )
        delta_pos, delta_vel = remove_com_and_angular_momentum(mol, delta_pos, delta_vel)
        mol.set_positions(mol.get_positions() + delta_pos)
        mol.set_velocities(delta_vel / units.fs)
        mol.info['zpe_report'] = verify_zpe(
            self.template,
            delta_pos,
            delta_vel,
            self.freqs_cm1,
            self.eig_mass,
            self.masses_dof,
            self.v_quantum,
        )
        return mol


class MoleculeRotationInitializer:
    def __init__(self, temperature_K, seed_offset: int = 0):
        self.temperature_K = rotational_temperature_to_kelvin(temperature_K)
        self.seed_offset = seed_offset

    @property
    def enabled(self) -> bool:
        return self.temperature_K > 0.0

    def sample(self, mol: Atoms, seed: int) -> float:
        if not self.enabled:
            return 0.0
        return add_rotational_energy_from_temperature(
            mol,
            temperature_K=self.temperature_K,
            seed=seed + self.seed_offset,
        )


In [ ]:
# --- Surface and placement helpers ---

def random_rotation_matrix(rng: np.random.Generator) -> np.ndarray:
    u1, u2, u3 = rng.random(3)
    qx = np.sqrt(1.0 - u1) * np.sin(2 * np.pi * u2)
    qy = np.sqrt(1.0 - u1) * np.cos(2 * np.pi * u2)
    qz = np.sqrt(u1) * np.sin(2 * np.pi * u3)
    qw = np.sqrt(u1) * np.cos(2 * np.pi * u3)
    return np.array([
        [1 - 2 * (qy**2 + qz**2), 2 * (qx * qy - qz * qw), 2 * (qx * qz + qy * qw)],
        [2 * (qx * qy + qz * qw), 1 - 2 * (qx**2 + qz**2), 2 * (qy * qz - qx * qw)],
        [2 * (qx * qz - qy * qw), 2 * (qy * qz + qx * qw), 1 - 2 * (qx**2 + qy**2)],
    ])


def rotate_atoms_randomly(atoms: Atoms, seed: int) -> None:
    rng = np.random.default_rng(seed)
    R = random_rotation_matrix(rng)
    com = atoms.get_center_of_mass()
    rel = atoms.get_positions() - com
    atoms.set_positions(com + rel @ R.T)
    vel = atoms.get_velocities()
    if vel is not None:
        atoms.set_velocities(vel @ R.T)


def fractional_xy_to_cart(cell: np.ndarray, frac_xy: Sequence[float]) -> np.ndarray:
    frac = np.array([frac_xy[0], frac_xy[1], 0.0])
    cart = frac @ cell
    return cart[:2]


def choose_xy_target(slab: Atoms, mode: str, rng: np.random.Generator, frac_xy: Optional[Sequence[float]] = None) -> Tuple[np.ndarray, np.ndarray]:
    if mode == "center" or frac_xy is None:
        xy = slab.positions[:, :2].mean(axis=0)
        frac = np.array([0.5, 0.5])
    else:
        frac = np.array(frac_xy, dtype=float)
        xy = fractional_xy_to_cart(slab.cell, frac)
    return xy, frac


def place_molecule_above_surface(mol: Atoms, slab: Atoms, height: float, xy_target: np.ndarray) -> None:
    z_top = slab.positions[:, 2].max()
    target_com = np.array([xy_target[0], xy_target[1], z_top + height])
    shift = target_com - mol.get_center_of_mass()
    mol.positions += shift


def wrap_molecule_by_com_xy(mol: Atoms, cell: np.ndarray) -> None:
    com = mol.get_center_of_mass()
    frac = np.linalg.solve(np.asarray(cell).T, com)
    shift_frac = np.array([-np.floor(frac[0]), -np.floor(frac[1]), 0.0])
    mol.positions += shift_frac @ np.asarray(cell)


def add_incident_energy(mol: Atoms, incident_energy_eV: float, direction: Sequence[float]) -> float:
    if incident_energy_eV < 0:
        raise ValueError("incident_energy_eV must be non-negative.")
    if incident_energy_eV == 0.0:
        return 0.0
    direction = np.asarray(direction, dtype=float)
    norm = np.linalg.norm(direction)
    if norm == 0.0:
        raise ValueError("incident_direction cannot be the zero vector.")
    unit_dir = direction / norm
    total_mass_amu = mol.get_masses().sum()
    speed_ang_per_fs = np.sqrt(2.0 * incident_energy_eV / (total_mass_amu * KCONV))
    delta_v = speed_ang_per_fs * unit_dir
    vel = mol.get_velocities()
    if vel is None:
        vel = np.zeros((len(mol), 3))
    mol.set_velocities(vel + delta_v / units.fs)
    return speed_ang_per_fs


def min_intersection_distance(slab: Atoms, mol: Atoms) -> float:
    slab_pos = slab.get_positions()
    mol_pos = mol.get_positions()
    diff = slab_pos[:, None, :] - mol_pos[None, :, :]
    dist = np.linalg.norm(diff, axis=2)
    return float(dist.min())


def enforce_clearance(slab: Atoms, mol: Atoms, min_clearance: float, lift_step: float = 0.2) -> float:
    clearance = min_intersection_distance(slab, mol)
    if clearance >= min_clearance:
        return clearance
    lift = max(0.0, min_clearance - clearance + lift_step)
    mol.positions[:, 2] += lift
    return min_intersection_distance(slab, mol)


def _read_vasp_xml_velocities(path: Path) -> list[np.ndarray]:
    velocities = []
    calculation_stack = []
    try:
        for event, elem in ET.iterparse(path, events=["start", "end"]):
            if event == "start" and elem.tag == "calculation":
                calculation_stack.append(elem)
            elif event == "end" and elem.tag == "calculation":
                step = calculation_stack.pop() if calculation_stack else elem
                vblock = step.find("varray[@name='velocities']")
                if vblock is not None:
                    vel = np.array([
                        [float(val) for val in row.text.split()]
                        for row in vblock
                    ], dtype=float)
                    velocities.append(vel)
                elem.clear()
    except ET.ParseError:
        pass
    return velocities


def _resolve_indices(index, n_frames: int) -> list[int]:
    if isinstance(index, slice):
        return list(range(n_frames))[index]
    if isinstance(index, int):
        return [index if index >= 0 else n_frames + index]
    if isinstance(index, str):
        if ":" in index:
            parts = index.split(":")
            if len(parts) > 3:
                raise ValueError("Invalid surface index.")
            values = [int(part) if part else None for part in parts]
            values += [None] * (3 - len(values))
            return list(range(n_frames))[slice(*values)]
        idx = int(index)
        return [idx if idx >= 0 else n_frames + idx]
    raise TypeError("Unsupported surface index type.")


def _estimate_velocities_from_positions(frames: Sequence[Atoms], potim_fs: float) -> list[np.ndarray]:
    if potim_fs <= 0:
        raise ValueError("POTIM_fs must be strictly positive to estimate velocities.")
    if len(frames) == 0:
        return []
    if len(frames) == 1:
        return [np.zeros((len(frames[0]), 3))]

    frac_positions = [atoms.get_scaled_positions(wrap=False) for atoms in frames]
    cells = [np.asarray(atoms.cell) for atoms in frames]
    pbc = np.asarray(frames[0].pbc, dtype=bool)
    velocities = []

    for idx in range(len(frames)):
        if idx == 0:
            frac_delta = frac_positions[1] - frac_positions[0]
            cell = cells[0]
            dt_fs = potim_fs
        elif idx == len(frames) - 1:
            frac_delta = frac_positions[-1] - frac_positions[-2]
            cell = cells[-1]
            dt_fs = potim_fs
        else:
            frac_delta = frac_positions[idx + 1] - frac_positions[idx - 1]
            cell = 0.5 * (cells[idx + 1] + cells[idx - 1])
            dt_fs = 2.0 * potim_fs

        frac_delta[:, pbc] -= np.round(frac_delta[:, pbc])
        cart_delta = frac_delta @ cell
        velocities.append(cart_delta / dt_fs)
    return velocities


def load_surface_ensemble(path: Path, fmt: Optional[str], index: str, keep_velocities: bool = True, potim_fs: float = 1.0):
    frames = read(path, index=index, format=fmt)
    if isinstance(frames, Atoms):
        frames = [frames]
    else:
        frames = list(frames)

    if not keep_velocities:
        return frames

    velocities_all = _read_vasp_xml_velocities(path)
    if velocities_all:
        selected = _resolve_indices(index, len(velocities_all))
        if len(selected) != len(frames):
            raise ValueError("The number of ASE snapshots does not match the number of velocity blocks extracted from vasprun.xml.")
    else:
        all_frames = read(path, index=":", format=fmt)
        if isinstance(all_frames, Atoms):
            all_frames = [all_frames]
        else:
            all_frames = list(all_frames)
        velocities_all = _estimate_velocities_from_positions(all_frames, potim_fs=potim_fs)
        selected = _resolve_indices(index, len(all_frames))
        if len(selected) != len(frames):
            raise ValueError("The number of selected snapshots does not match the subset used to estimate surface velocities.")

    for atoms, vel_idx in zip(frames, selected):
        vel = velocities_all[vel_idx]
        if vel.shape != (len(atoms), 3):
            raise ValueError("The shape of the read velocities does not match the selected surface snapshot.")
        # VASP velocity blocks are in Angstrom/fs; ASE stores velocities in Angstrom/ASE-time.
        # Dividing by units.fs preserves the intended surface temperature when ASE writes POSCAR.
        atoms.set_velocities(vel / units.fs)
    return frames


def merge_slab_and_molecule(slab: Atoms, mol: Atoms) -> Atoms:
    system = slab + mol
    vel = np.zeros((len(system), 3))
    if slab.get_velocities() is not None:
        vel[:len(slab)] = slab.get_velocities()
    if mol.get_velocities() is not None:
        vel[len(slab):] = mol.get_velocities()
    system.set_velocities(vel)
    return system

In [ ]:
# --- Input loading ---
molecule_eq = load_structure_from_vasprun(PATHS["molecule_vasprun"])
freqs_cm1, eigvecs_cart = load_modes_from_vasprun(PATHS["molecule_vasprun"])
save_modes_npz(PATHS["vasp_modes_npz"], freqs_cm1, eigvecs_cart)

rotation_initializer = MoleculeRotationInitializer(
    temperature_K=ROTATION_SETTINGS["temperature_K"],
    seed_offset=ROTATION_SETTINGS["seed_offset"],
)

zpe_initializer = None
if ZPE_SETTINGS["enabled"]:
    zpe_initializer = MoleculeZPEInitializer(
        molecule_eq=molecule_eq,
        freqs_cm1=freqs_cm1,
        eigvecs_cart=eigvecs_cart,
        v_quantum=ZPE_SETTINGS["v_quantum"],
        seed_offset=ZPE_SETTINGS["seed_offset"],
    )

surface_ensemble = load_surface_ensemble(
    PATHS["surface_vasprun"],
    fmt=SURFACE_SAMPLING["format"],
    index=SURFACE_SAMPLING["index"],
    keep_velocities=SURFACE_SAMPLING["keep_velocities"],
    potim_fs=SURFACE_SAMPLING["POTIM_fs"],
)

print(f"Molecule: {len(molecule_eq)} atoms")
print(f"Surface: {len(surface_ensemble)} snapshots loaded")
print(f"Modes loaded from vasprun.xml :: {PATHS['molecule_vasprun']}")
print(f".npz cache written :: {PATHS['vasp_modes_npz']}")
print(f"ZPE enabled: {ZPE_SETTINGS['enabled']}")
print(f"Rotational temperature: {rotation_initializer.temperature_K:.3f} K")

In [ ]:
# --- POSCAR batch generation ---
def format_incident_energy_label(energy_eV: float) -> str:
    return f"Ei{float(energy_eV):g}"


def infer_surface_temperature_label(surface_path: Path) -> str:
    match = re.search(r"(\d+(?:\.\d+)?)K", str(surface_path))
    if match is None:
        raise ValueError(
            f"Cannot infer the surface temperature from {surface_path}. "
            "Include a label such as 100K, 300K, or 500K in the path."
        )
    return f"{float(match.group(1)):g}K"


surface_temperature_label = infer_surface_temperature_label(PATHS["surface_vasprun"])
incident_energy_label = format_incident_energy_label(GENERATION["incident_energy_eV"])
run_output_dir = PATHS["output_dir"] / surface_temperature_label / incident_energy_label
run_output_dir.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(GENERATION["master_seed"])
metadata = []

for idx in range(GENERATION["n_configurations"]):
    mol_seed = int(rng.integers(0, 2**32 - 1))
    orient_seed = int(rng.integers(0, 2**32 - 1))
    frac_xy = rng.random(2) if GENERATION["xy_mode"] == "random" else np.array([0.5, 0.5])
    snapshot_idx = int(rng.integers(0, len(surface_ensemble)))

    slab = surface_ensemble[snapshot_idx].copy()
    if not SURFACE_SAMPLING["keep_velocities"] or slab.get_velocities() is None:
        slab.set_velocities(np.zeros((len(slab), 3)))

    rotation_energy_eV = 0.0
    zpe_report = None
    if ZPE_SETTINGS["enabled"]:
        mol = zpe_initializer.sample(seed=mol_seed)
        zpe_report = mol.info.get("zpe_report")
    else:
        mol = molecule_eq.copy()
        mol.set_velocities(np.zeros((len(mol), 3)))
    if rotation_initializer.enabled:
        rotation_energy_eV = rotation_initializer.sample(mol, seed=mol_seed)
    rotate_atoms_randomly(mol, orient_seed)

    xy_target, frac_used = choose_xy_target(slab, GENERATION["xy_mode"], rng, frac_xy)
    place_molecule_above_surface(mol, slab, height=GENERATION["surface_distance_A"], xy_target=xy_target)
    clearance = enforce_clearance(slab, mol, GENERATION["min_clearance"])
    wrap_molecule_by_com_xy(mol, slab.cell)
    incident_speed = add_incident_energy(
        mol,
        incident_energy_eV=GENERATION["incident_energy_eV"],
        direction=GENERATION["incident_direction"],
    )

    system = merge_slab_and_molecule(slab, mol)
    tag = f"config_{idx:03d}"
    # poscar_path = PATHS["output_dir"] / f"POSCAR_{idx:03d}"
    poscar_path = run_output_dir / f"POSCAR-{idx + 1}"
    write(poscar_path, system, format="vasp", vasp5=True, sort=False, direct=False)

    metadata.append({
        "tag": tag,
        "poscar": str(poscar_path),
        "surface_temperature_label": surface_temperature_label,
        "surface_vasprun": str(PATHS["surface_vasprun"]),
        "incident_energy_label": incident_energy_label,
        "snapshot_index": snapshot_idx,
        "mol_seed": mol_seed,
        "orientation_seed": orient_seed,
        "xy_fractional": frac_used.tolist(),
        "surface_distance_A": GENERATION["surface_distance_A"],
        "clearance_A": clearance,
        "incident_energy_eV": GENERATION["incident_energy_eV"],
        "incident_speed_A_fs": incident_speed,
        "rotational_temperature_K": rotation_initializer.temperature_K,
        "rotational_energy_eV": rotation_energy_eV,
        "zpe_target_total_eV": None if zpe_report is None else zpe_report["target_total_eV"],
        "zpe_modal_kinetic_eV": None if zpe_report is None else zpe_report["modal_kinetic_eV"],
        "zpe_modal_potential_eV": None if zpe_report is None else zpe_report["modal_potential_eV"],
        "zpe_modal_total_eV": None if zpe_report is None else zpe_report["modal_total_eV"],
        "zpe_external_leak_eV": None if zpe_report is None else zpe_report["external_leak_eV"],
        "zpe_mode_details": None if zpe_report is None else zpe_report["mode_details"],
    })

meta_path = run_output_dir / "metadata.json"
meta_path.write_text(json.dumps(metadata, indent=2))
print(f"{len(metadata)} configurations written to {run_output_dir}")
print(f"Metadata : {meta_path}")

## Initial-Condition Validation


In [ ]:
# --- Recursive validation of generated POSCAR files ---
# Define the folder containing the initial conditions to validate.
# Examples:
#   VALIDATION_ROOT = Path("outputs/qct_poscars")
#   VALIDATION_ROOT = Path("outputs/qct_poscars_hpc_SO2")
VALIDATION_ROOT = Path("outputs/qct_poscars_hpc_SO2")

# This cell rereads the velocity blocks written in the POSCAR files,
# then checks the instantaneous surface temperature and molecule COM incident energy.
from collections import defaultdict
from ase.data import atomic_masses, atomic_numbers


def _parse_poscar_velocity_block(path: Path):
    lines = path.read_text().splitlines()
    symbols = lines[5].split()
    counts = [int(x) for x in lines[6].split()]
    atom_symbols = []
    for symbol, count in zip(symbols, counts):
        atom_symbols.extend([symbol] * count)
    natoms = sum(counts)

    velocity_start = None
    for line_idx in range(8, len(lines)):
        text = lines[line_idx].strip().lower()
        if text.startswith(("cartesian", "direct")) and line_idx > 8 + natoms:
            velocity_start = line_idx + 1
            break
    if velocity_start is None:
        raise ValueError(f"No velocity block found in {path}")

    velocities = np.array([
        [float(value) for value in lines[line_idx].split()[:3]]
        for line_idx in range(velocity_start, velocity_start + natoms)
    ])
    masses = np.array([atomic_masses[atomic_numbers[symbol]] for symbol in atom_symbols])
    return masses, velocities


def _kinetic_energy_eV_from_vasp_velocities(masses_amu, velocities_A_fs):
    return 0.5 * float(np.sum(masses_amu[:, None] * velocities_A_fs**2)) * KCONV


def _temperature_K_from_vasp_velocities(masses_amu, velocities_A_fs, remove_com=True):
    active = np.linalg.norm(velocities_A_fs, axis=1) > 1e-12
    masses = masses_amu[active]
    velocities = velocities_A_fs[active].copy()
    dof = 3 * len(masses)
    if remove_com and len(masses) > 1:
        velocities -= np.average(velocities, axis=0, weights=masses)
        dof -= 3
    if dof <= 0:
        return float("nan")
    kinetic_eV = _kinetic_energy_eV_from_vasp_velocities(masses, velocities)
    return 2.0 * kinetic_eV / (dof * KB_EV_K)


def _surface_temperature_from_path(path: Path):
    text = str(path)
    match = re.search(r"Ts(\d+(?:\.\d+)?)", text)
    if match is None:
        match = re.search(r"(\d+(?:\.\d+)?)K", text)
    return None if match is None else float(match.group(1))


def _incident_energy_from_path(path: Path):
    match = re.search(r"Ei(\d+(?:\.\d+)?)", str(path))
    return None if match is None else float(match.group(1))


poscar_paths = sorted(VALIDATION_ROOT.glob("**/POSCAR-*"))
if not poscar_paths:
    raise FileNotFoundError(f"No POSCAR-* files found recursively in {VALIDATION_ROOT}")

surface_natoms = len(surface_ensemble[0])
molecule_natoms = len(molecule_eq)
records = []

for poscar_path in poscar_paths:
    masses, velocities = _parse_poscar_velocity_block(poscar_path)
    if len(masses) != surface_natoms + molecule_natoms:
        raise ValueError(f"Unexpected atom count in {poscar_path}: {len(masses)}")

    surface_temperature = _temperature_K_from_vasp_velocities(
        masses[:surface_natoms], velocities[:surface_natoms], remove_com=True
    )
    mol_masses = masses[-molecule_natoms:]
    mol_velocities = velocities[-molecule_natoms:]
    mol_vcom = np.average(mol_velocities, axis=0, weights=mol_masses)
    incident_energy = 0.5 * mol_masses.sum() * mol_vcom[2]**2 * KCONV

    records.append({
        "path": poscar_path,
        "target_surface_temperature_K": _surface_temperature_from_path(poscar_path),
        "surface_temperature_K": surface_temperature,
        "target_incident_energy_eV": _incident_energy_from_path(poscar_path),
        "incident_energy_eV": incident_energy,
    })

by_temperature = defaultdict(list)
by_incident_energy = defaultdict(list)
for record in records:
    by_temperature[record["target_surface_temperature_K"]].append(record)
    by_incident_energy[record["target_incident_energy_eV"]].append(record)

print(f"Validated folder: {VALIDATION_ROOT}")
print(f"POSCAR files found recursively: {len(records)}")

print("\nSurface temperatures:")
for target_temp, group in sorted(by_temperature.items(), key=lambda item: (-1 if item[0] is None else item[0])):
    values = np.array([record["surface_temperature_K"] for record in group])
    label = "unknown" if target_temp is None else f"{target_temp:g} K"
    print(f"  Ts {label}: n={len(group)}, mean={values.mean():.1f} K, range=[{values.min():.1f}, {values.max():.1f}] K")

print("\nMolecule COM incident energies:")
max_ei_error = 0.0
for target_ei, group in sorted(by_incident_energy.items(), key=lambda item: (-1 if item[0] is None else item[0])):
    values = np.array([record["incident_energy_eV"] for record in group])
    label = "unknown" if target_ei is None else f"{target_ei:g} eV"
    if target_ei is None:
        error = float("nan")
    else:
        error = float(np.max(np.abs(values - target_ei)))
        max_ei_error = max(max_ei_error, error)
    print(f"  Ei {label}: n={len(group)}, mean={values.mean():.6f} eV, max error={error:.2e} eV")

for target_ei, group in by_incident_energy.items():
    if target_ei is None:
        continue
    values = np.array([record["incident_energy_eV"] for record in group])
    if np.max(np.abs(values - target_ei)) > 1e-6:
        raise ValueError(f"Validation failed: inconsistent Ei for {target_ei} eV")

for target_temp, group in by_temperature.items():
    if target_temp is None:
        continue
    values = np.array([record["surface_temperature_K"] for record in group])
    if not (0.3 * target_temp <= values.mean() <= 1.7 * target_temp):
        raise ValueError(f"Validation failed: inconsistent mean temperature for Ts {target_temp} K")

print("\nValidation complete: temperatures and energies are consistent.")
